# Data Preparation for MedQA Uncertainty Experiment

This notebook prepares a clean dataset from MedQA JSONL by cleaning vignettes, splitting information layers with an LLM, caching progress, and exporting a validated JSON file.

## Section 1 - Setup and Config

In [1]:
import json
import logging
import os
import random
import re
import time
from pathlib import Path

from google import genai
from google.genai.types import GenerateContentConfig, HttpOptions
from tenacity import retry, stop_after_attempt, wait_exponential

# Centralized configuration (edit only this block).
JSONL_PATH = "train.jsonl"
OUTPUT_PATH = "outputs/medqa_prepared.json"
SPLITTER_INSTRUCTION_PATH = "splitter_instruction.txt"
N_RECORDS = 20
RANDOM_SEED = 42
MODEL_ID = "gemini-2.5-flash"
TEMPERATURE = 0.0
REQUEST_INTERVAL = 2.0
CACHE_PATH = "outputs/split_cache.json"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("medqa_prep")

# Load .env line by line and set env vars only if not already set.
env_path = Path(".env")
if env_path.exists():
    with env_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())

if "VERTEX_API_KEY" not in os.environ:
    raise RuntimeError("VERTEX_API_KEY not found in environment or .env file.")

client = genai.Client(
    api_key=os.environ["VERTEX_API_KEY"],
    http_options=HttpOptions(api_version="v1"),
)

logger.info("Configuration and client initialization complete.")

2026-04-26 21:39:23,373 | INFO | Configuration and client initialization complete.


## Section 2 - Load and Sample

In [2]:
records = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

total_records = len(records)
if total_records == 0:
    raise RuntimeError(f"No records found in {JSONL_PATH}.")
if N_RECORDS <= 0:
    raise ValueError("N_RECORDS must be > 0.")

sample_n = min(N_RECORDS, total_records)
rng = random.Random(RANDOM_SEED)
sampled_records = rng.sample(records, sample_n)

logger.info("Loaded %d records and sampled %d records.", total_records, sample_n)

# Explicit display output
print(f"Total records loaded: {total_records}")
print(f"Total records sampled: {sample_n}")
for i, rec in enumerate(sampled_records[:2], 1):
    print("\n" + "=" * 100)
    print(f"Sample raw record {i}")
    print(json.dumps(rec, indent=2, ensure_ascii=False))

2026-04-26 21:39:25,904 | INFO | Loaded 10178 records and sampled 20 records.


Total records loaded: 10178
Total records sampled: 20

Sample raw record 1
{
  "question": "An 8-year-old African-American boy is brought to the emergency room with severe pain in both hands. His mother says that the patient had a fever with a cough a couple of days ago. Family history is positive for an uncle who died from a blood disease. A peripheral blood smear of this patient is shown in the image. Which of the following is the most likely mechanism for this patient’s disease?",
  "answer": "Missense mutation",
  "options": {
    "A": "Nonsense mutation",
    "B": "Frameshift mutation",
    "C": "Mismatch repair",
    "D": "Silent mutation",
    "E": "Missense mutation"
  },
  "meta_info": "step1",
  "answer_idx": "E"
}

Sample raw record 2
{
  "question": "An 11-year-old boy is brought to the emergency department by his parents with a 2-day history of fever, malaise, and productive cough. On presentation, he is found to be very weak and is having difficulty breathing. His past me

## Section 2.5 - Filter Non-Clinical-Reasoning Questions

Remove records where the question asks about molecular mechanisms, pathophysiology, embryology, or lab marker patterns rather than diagnosis or treatment. These are knowledge-recall questions and do not work for the uncertainty experiment.

A record is kept only if the question stem asks for a diagnosis, treatment, next best step, or management decision.

In [4]:
# Patterns that indicate a CLINICAL REASONING question — keep these
clinical_terms = [
    "most likely diagnosis",
    "most likely cause",
    "most likely responsible",
    "best treatment",
    "best next step",
    "next best step",
    "next step in management",
    "best initial",
    "best management",
    "should be treated",
    "most appropriate treatment",
    "most appropriate management",
    "most appropriate next",
    "drug of choice",
    "most likely to (?:be|have|develop|cause|result)",
    "precautions could have prevented",
    "most likely to improve",
    "best course of action",
]
CLINICAL_PATTERNS = re.compile("|".join(clinical_terms), flags=re.IGNORECASE)

# Patterns that indicate a KNOWLEDGE RECALL question — exclude these
recall_terms = [
    "mechanism",
    "mutation",
    "embryolog",
    "most likely defective",
    "most likely derived from",
    "most likely involved",
    "most likely responsible for the (?:mechanism|pathophysiology|genetic)",
    "which of the following (?:proteins|genes|enzymes|receptors|pathways|structures|cells)",
    "most likely (?:pathway|protein|gene|enzyme|receptor|structure|cell type)",
    "lab(?:oratory)? (?:findings|results|values|markers)\\s+would",
    "serologic",
    "serological",
    "immunofluorescence",
    "histopatholog",
    "embryologic",
    "precursor cell",
    "cell of origin",
]
RECALL_PATTERNS = re.compile("|".join(recall_terms), flags=re.IGNORECASE)

def is_clinical_reasoning(record):
    """
    Returns True if the question is a clinical reasoning question
    (diagnosis, treatment, management, next step).
    Returns False for pure knowledge-recall questions.
    """
    question = record.get("question", "")
    # If it matches a recall pattern, exclude regardless of clinical pattern.
    if RECALL_PATTERNS.search(question):
        return False
    # If it matches a clinical pattern, include.
    if CLINICAL_PATTERNS.search(question):
        return True
    # Default: include if we cannot determine — we will flag it.
    return True

before_count = len(sampled_records)
excluded_records = [r for r in sampled_records if not is_clinical_reasoning(r)]
sampled_records = [r for r in sampled_records if is_clinical_reasoning(r)]
after_count = len(sampled_records)

logger.info(
    "Filtering: %d records before, %d after, %d excluded.",
    before_count, after_count, len(excluded_records)
)

# Explicit display output
print(f"Records before filtering: {before_count}")
print(f"Records after filtering:  {after_count}")
print(f"Records excluded:         {len(excluded_records)}")

if excluded_records:
    print("\nExcluded records (question stem shown):")
    for rec in excluded_records:
        q = rec.get("question", "")[-200:]
        print(f"  - {q}")

print("\nNOTE: If too many records were excluded, increase N_RECORDS in Section 1 config")
print("      so that after filtering you still have enough records for the pilot.")

2026-04-26 21:41:36,222 | INFO | Filtering: 20 records before, 13 after, 7 excluded.


Records before filtering: 20
Records after filtering:  13
Records excluded:         7

Excluded records (question stem shown):
  -  is positive for an uncle who died from a blood disease. A peripheral blood smear of this patient is shown in the image. Which of the following is the most likely mechanism for this patient’s disease?
  - gs include decreased levels of IgG, IgM, IgA, and plasma cells with normal levels of CD4 positive cells. The protein that is most likely defective in this patient has which of the following functions?
  -  several dysplastic nevi removed in the past. A photograph of the lesion is shown. The lesion is most likely derived from cells that are also the embryological origin of which of the following tumors?
  -  to assess the patient’s serologic status:
HBV DNA positive
HBsAg negative
HBeAg  negative
HBsAb negative
HBcAb positive
HBeAb negative
Which of the following disease states is the patient exhibiting?
  - l bilaterally. She is started on intravenous flui

## Section 3 - Vignette Cleaning

In [5]:
QUESTION_STEM_RE = re.compile(
    r"\s*(?:Which of the following|What is the most likely|What is the best|What is the|What would be|The next best step|How should)\b[\s\S]*$",
    flags=re.IGNORECASE,
)

def clean_vignette(question_text):
    text = str(question_text or "").strip()
    cleaned = QUESTION_STEM_RE.sub("", text).strip()
    return cleaned if cleaned else text

for rec in sampled_records:
    rec["vignette"] = clean_vignette(rec.get("question", ""))

logger.info("Vignette cleaning complete for %d sampled records.", len(sampled_records))

# Explicit display output
for i, rec in enumerate(sampled_records[:3], 1):
    print("\n" + "=" * 100)
    print(f"Example {i} - Original question:")
    print(rec.get("question", ""))
    print("\nExample {i} - Cleaned vignette:")
    print(rec.get("vignette", ""))

2026-04-26 21:41:47,732 | INFO | Vignette cleaning complete for 13 sampled records.



Example 1 - Original question:
A 37-year-old woman presents for prenatal counseling at 18 weeks gestation. The patient tells you that her sister recently had a child with Down's syndrome, and the patient would like prenatal screening for Down's in her current pregnancy.

Which of the following prenatal screening tests and results would raise concern for Down's syndrome?

Example {i} - Cleaned vignette:
A 37-year-old woman presents for prenatal counseling at 18 weeks gestation. The patient tells you that her sister recently had a child with Down's syndrome, and the patient would like prenatal screening for Down's in her current pregnancy.

Example 2 - Original question:
A 73-year-old man presents to the office, complaining of “weird blisters” on his right hand, which appeared 2 weeks ago. The patient says that he initially had a rash, which progressed to blisters. He denies any trauma or known contact with sick people. He is worried because he hasn’t been able to garden since the rash 

## Section 4 - LLM Splitting with Cache

In [6]:
def strip_code_fences(text):
    txt = text.strip()
    if txt.startswith("```"):
        txt = re.sub(r"^```[a-zA-Z0-9_\-]*\n?", "", txt)
        txt = re.sub(r"\n?```$", "", txt)
    return txt.strip()

def parse_split_json(response_text, original_vignette):
    cleaned = strip_code_fences(response_text)
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        logger.warning("Failed to parse splitter JSON: %s", e)
        return {"layer_0": original_vignette, "layer_1": ""}

    if not isinstance(parsed, dict):
        logger.warning("Splitter output was not a dict. Using fallback.")
        return {"layer_0": original_vignette, "layer_1": ""}

    return {
        "layer_0": str(parsed.get("layer_0", "")).strip() or original_vignette,
        "layer_1": str(parsed.get("layer_1", "")).strip(),
    }

with open(SPLITTER_INSTRUCTION_PATH, "r", encoding="utf-8") as f:
    splitter_instruction = f.read().strip()

@retry(wait=wait_exponential(multiplier=1, min=1, max=20), stop=stop_after_attempt(6), reraise=True)
def split_vignette(vignette):
    prompt = f"{splitter_instruction}\n\nVIGNETTE:\n{vignette}"
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=GenerateContentConfig(temperature=TEMPERATURE),
    )

    parts = response.candidates[0].content.parts
    non_thinking_text = []
    for part in parts:
        if getattr(part, "thought", False):
            continue
        part_text = getattr(part, "text", "")
        if part_text:
            non_thinking_text.append(part_text)

    response_text = "\n".join(non_thinking_text).strip()
    if not response_text:
        logger.warning("Empty non-thinking response. Using fallback.")
        return {"layer_0": vignette, "layer_1": ""}

    return parse_split_json(response_text, vignette)

cache_path = Path(CACHE_PATH)
cache_path.parent.mkdir(parents=True, exist_ok=True)
if cache_path.exists():
    with cache_path.open("r", encoding="utf-8") as f:
        split_cache = json.load(f)
    logger.info("Loaded cache with %d entries from %s", len(split_cache), CACHE_PATH)
else:
    split_cache = {}
    logger.info("No existing cache found. Starting with empty cache.")

def make_cache_key(question_text):
    return str(question_text or "")[:200]

for i, rec in enumerate(sampled_records, 1):
    question_text = rec.get("question", "")
    key = make_cache_key(question_text)

    if key in split_cache:
        split_result = split_cache[key]
        status = "cached"
    else:
        try:
            split_result = split_vignette(rec.get("vignette", ""))
        except Exception as e:
            logger.warning("split_vignette failed after retries: %s", e)
            split_result = {"layer_0": rec.get("vignette", ""), "layer_1": ""}

        split_cache[key] = split_result
        with cache_path.open("w", encoding="utf-8") as f:
            json.dump(split_cache, f, ensure_ascii=False, indent=2)
        status = "fresh"

    rec["layer_0"] = str(split_result.get("layer_0", "")).strip()
    rec["layer_1"] = str(split_result.get("layer_1", "")).strip()

    # Explicit display output
    l0_preview = rec["layer_0"][:80].replace("\n", " ")
    l1_preview = rec["layer_1"][:80].replace("\n", " ")
    print(
        f"[{i}/{len(sampled_records)}] {status} | layer_0: {l0_preview} | layer_1: {l1_preview}"
    )

    if i < len(sampled_records):
        time.sleep(REQUEST_INTERVAL)

logger.info("Splitting phase completed for %d sampled records.", len(sampled_records))

2026-04-26 21:42:20,239 | INFO | No existing cache found. Starting with empty cache.
2026-04-26 21:42:20,243 | INFO | AFC is enabled with max remote calls: 10.
2026-04-26 21:42:24,635 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


[1/13] fresh | layer_0: A 37-year-old woman presents for prenatal counseling at 18 weeks gestation. The  | layer_1: 


2026-04-26 21:42:26,643 | INFO | AFC is enabled with max remote calls: 10.
2026-04-26 21:42:29,755 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


[2/13] fresh | layer_0: A 73-year-old man presents to the office, complaining of “weird blisters” on his | layer_1: His vital signs are stable. On physical exam, the patient has multiple bullae ac


2026-04-26 21:42:31,761 | INFO | AFC is enabled with max remote calls: 10.
2026-04-26 21:42:36,188 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


...

[11/13] fresh | layer_0: A 54-year-old man was brought to the emergency room due to acute onset of slurre | layer_1: His blood pressure is 90/50 mm Hg, respiratory rate is 12/min, and heart rate is


2026-04-26 21:43:47,777 | INFO | AFC is enabled with max remote calls: 10.
2026-04-26 21:43:52,826 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


[12/13] fresh | layer_0: A 25-year-old woman, gravida 2, para 1, at 25 weeks' gestation comes to the emer | layer_1: Her temperature is 39°C (102.2°F), pulse is 110/min, respirations are 20/min, an


2026-04-26 21:43:54,835 | INFO | AFC is enabled with max remote calls: 10.
2026-04-26 21:43:57,608 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-26 21:43:57,614 | INFO | Splitting phase completed for 13 sampled records.


[13/13] fresh | layer_0: A 52-year-old man presents his primary care physician for follow-up. 3 months ag | layer_1: Today, his HbA1C is 7.9%. The physician decides to add pioglitazone for better c


## Section 5 - Assemble and Save

In [7]:
prepared_records = []
for idx, rec in enumerate(sampled_records):
    prepared_records.append({
        "id": f"medqa_{idx}",
        "vignette_full": rec.get("vignette", ""),
        "layer_0": rec.get("layer_0", ""),
        "layer_1": rec.get("layer_1", ""),
        "answer_choices": rec.get("options", {}),
        "correct_answer_text": rec.get("answer", ""),
        "correct_answer_idx": rec.get("answer_idx", ""),
        "meta_info": rec.get("meta_info", ""),
    })

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(prepared_records, f, ensure_ascii=False, indent=2)

logger.info("Saved %d prepared records to %s", len(prepared_records), OUTPUT_PATH)

# Explicit display output
print(f"Total prepared records saved: {len(prepared_records)}")
for i, rec in enumerate(prepared_records[:2], 1):
    print("\n" + "=" * 100)
    print(f"Prepared record {i}")
    print(json.dumps(rec, indent=2, ensure_ascii=False))

2026-04-26 21:44:53,422 | INFO | Saved 13 prepared records to outputs/medqa_prepared.json


Total prepared records saved: 13

Prepared record 1
{
  "id": "medqa_0",
  "vignette_full": "A 37-year-old woman presents for prenatal counseling at 18 weeks gestation. The patient tells you that her sister recently had a child with Down's syndrome, and the patient would like prenatal screening for Down's in her current pregnancy.",
  "layer_0": "A 37-year-old woman presents for prenatal counseling at 18 weeks gestation. The patient tells you that her sister recently had a child with Down's syndrome, and the patient would like prenatal screening for Down's in her current pregnancy.",
  "layer_1": "",
  "answer_choices": {
    "A": "Increased AFP, normal HCG, normal unconjugated estriol",
    "B": "Decreased AFP, increased HCG, decreased unconjugated estriol",
    "C": "Decreased AFP, decreased HCG, decreased unconjugated estriol",
    "D": "Normal AFP, increased HCG, decreased unconjugated estriol",
    "E": "Normal AFP, decreased HCG, decreased unconjugated estriol"
  },
  "correct_an

## Section 6 - Validation

In [8]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved_records = json.load(f)

required_fields = {
    "id",
    "vignette_full",
    "layer_0",
    "layer_1",
    "answer_choices",
    "correct_answer_text",
    "correct_answer_idx",
    "meta_info",
}
valid_answer_idx = {"A", "B", "C", "D", "E"}

missing_field_ids = []
empty_layer0_ids = []
empty_layer1_ids = []
invalid_answer_idx_ids = []

layer0_word_counts = []
layer1_word_counts = []

for rec in saved_records:
    rec_id = rec.get("id", "<missing-id>")

    if not required_fields.issubset(rec.keys()):
        missing_field_ids.append(rec_id)

    layer0 = str(rec.get("layer_0", "")).strip()
    layer1 = str(rec.get("layer_1", "")).strip()
    if not layer0:
        empty_layer0_ids.append(rec_id)
    if not layer1:
        empty_layer1_ids.append(rec_id)

    answer_idx = str(rec.get("correct_answer_idx", "")).strip()
    if answer_idx not in valid_answer_idx:
        invalid_answer_idx_ids.append(rec_id)

    layer0_word_counts.append(len(layer0.split()))
    layer1_word_counts.append(len(layer1.split()))

avg_layer0_words = sum(layer0_word_counts) / len(layer0_word_counts) if layer0_word_counts else 0
avg_layer1_words = sum(layer1_word_counts) / len(layer1_word_counts) if layer1_word_counts else 0

logger.info("Validation complete.")

# Explicit display output
print("Validation Summary")
print(f"Total records: {len(saved_records)}")
print(f"Records missing required fields: {len(missing_field_ids)}")
print(f"Records with empty layer_0: {len(empty_layer0_ids)}")
print(f"Records with empty layer_1: {len(empty_layer1_ids)}")
print(f"Records with invalid answer_idx: {len(invalid_answer_idx_ids)}")
print(f"Average layer_0 word count: {avg_layer0_words:.2f}")
print(f"Average layer_1 word count: {avg_layer1_words:.2f}")

if missing_field_ids:
    print("\nIDs missing required fields:", missing_field_ids)
if empty_layer0_ids:
    print("IDs with empty layer_0:", empty_layer0_ids)
if empty_layer1_ids:
    print("IDs with empty layer_1:", empty_layer1_ids)
if invalid_answer_idx_ids:
    print("IDs with invalid answer_idx:", invalid_answer_idx_ids)

2026-04-26 21:44:58,095 | INFO | Validation complete.


Validation Summary
Total records: 13
Records missing required fields: 0
Records with empty layer_0: 0
Records with empty layer_1: 1
Records with invalid answer_idx: 0
Average layer_0 word count: 45.38
Average layer_1 word count: 40.38
IDs with empty layer_1: ['medqa_0']


In [9]:
# Flag records where layer_0 is suspiciously thin (< 8 words)
# These are cases where the LLM splitter was too aggressive and the
# chief complaint got stripped down to almost nothing useful.
THIN_LAYER0_THRESHOLD = 8  # words

thin_records = [
    rec for rec in saved_records
    if len(str(rec.get('layer_0', '')).split()) < THIN_LAYER0_THRESHOLD
]

print(f"Records with thin layer_0 (< {THIN_LAYER0_THRESHOLD} words): {len(thin_records)}")
if thin_records:
    print("\nThin layer_0 records — consider excluding or re-splitting:")
    for rec in thin_records:
        print(f"  ID: {rec['id']}")
        print(f"    layer_0: {rec['layer_0']}")
        print(f"    layer_1 (first 100): {str(rec.get('layer_1',''))[:100]}")
        print()
else:
    print("All layer_0 fields have sufficient content. No action needed.")


Records with thin layer_0 (< 8 words): 0
All layer_0 fields have sufficient content. No action needed.
